<a href="https://colab.research.google.com/github/Shreya08-cyber/CSA6101-digital-forensics/blob/main/email_header_parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To parse raw email header metadata and check for discrepancies between key fields (like From, Return-Path, and standard SPF validation indicators) to identify potential email spoofing.

**Algorithm**

Load the raw text string containing the email header.

Parse the header fields using Python's built-in email package.

Extract critical sender metadata fields: From, Return-Path, and Received-SPF.

Extract the sending domain from the From address.

Extract the return domain from the Return-Path address.

Check if the From domain matches the Return-Path domain.

Inspect the Received-SPF header for pass/fail/softfail statuses.

Flag the email as potentially spoofed if domain mismatches or SPF failures are detected.

In [1]:
import email

raw_email_header = """From: Security Team <alert@yourbank.com>
Return-Path: <attacker@phishing-domain.xyz>
Received-SPF: softfail (google.com: domain of attacker@phishing-domain.xyz does not designate 192.0.2.1 as permitted sender)
Subject: Urgent: Verify Your Account
Date: Thu, 30 Jul 2026 10:15:00 +0000
"""

msg = email.message_from_string(raw_email_header)

from_header = msg.get("From", "")
return_path_header = msg.get("Return-Path", "")
spf_status = msg.get("Received-SPF", "")

from_domain = from_header.split("@")[-1].replace(">", "").strip() if "@" in from_header else ""
return_domain = return_path_header.split("@")[-1].replace(">", "").strip() if "@" in return_path_header else ""


domain_mismatch = (from_domain != return_domain)
spf_failed = "fail" in spf_status.lower() or "softfail" in spf_status.lower()

print("--- EMAIL SPOOFING ANALYSIS ---")
print(f"Header 'From' Domain:        {from_domain}")
print(f"Header 'Return-Path' Domain: {return_domain}")
print(f"SPF Record Status:          {spf_status.split()[0]}")
print("-" * 35)

if domain_mismatch or spf_failed:
    print("[WARNING] HIGH RISK: Sender spoofing indicators detected!")
    if domain_mismatch:
        print(" -> Mismatch detected between visible sender and actual reply path.")
    if spf_failed:
        print(" -> SPF policy check returned a softfail/fail status.")
else:
    print("[OK] Email headers pass basic authenticity checks.")

--- EMAIL SPOOFING ANALYSIS ---
Header 'From' Domain:        yourbank.com
Header 'Return-Path' Domain: phishing-domain.xyz
SPF Record Status:          softfail
-----------------------------------
[WARNING] HIGH RISK: Sender spoofing indicators detected!
 -> Mismatch detected between visible sender and actual reply path.
 -> SPF policy check returned a softfail/fail status.


**Result**

The script parses the headers and points out that the sender claims to be yourbank.com in the From field, but mail returns to phishing-domain.xyz. Flagged with an SPF softfail, the script triggers an explicit spoofing alert.